# NCAA Data Preparation
## Hypothesis: Does 3-Point Shooting Affect Team Success?

This notebook combines and cleans NCAA team data to explore the relationship between 3-point shooting (volume and efficiency) and team success (tournament performance, win rate, efficiency margin).

**Key sources:**
- `KenPom Barttorvik.csv` — per-year team efficiency, 3PT stats, tournament round reached
- `Shooting Splits.csv` — shot-type breakdown (dunks, close 2s, far 2s, 3s) with FG% and share

**Output:** `data/combined_ncaa.csv`

In [ ]:
import pandas as pd
import numpy as np
import os

DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), 'ncaa', 'data')
# If running notebook from the ncaa/ folder directly:
if not os.path.exists(DATA_DIR):
    DATA_DIR = os.path.join(os.getcwd(), 'data')

print('Data directory:', DATA_DIR)
print('Files available:', os.listdir(DATA_DIR))

## 1. Load Raw Data

In [ ]:
# Main per-year team stats: efficiency, 3PT shooting, tournament round, seed
kp = pd.read_csv(os.path.join(DATA_DIR, 'KenPom Barttorvik.csv'))

# Shot-type breakdown: 3s vs 2s vs dunks, offense and defense
ss = pd.read_csv(os.path.join(DATA_DIR, 'Shooting Splits.csv'))

print(f'KenPom Barttorvik: {kp.shape[0]} rows, {kp.shape[1]} cols')
print(f'Shooting Splits:   {ss.shape[0]} rows, {ss.shape[1]} cols')
print(f'\nYears in KenPom:   {sorted(kp["YEAR"].unique())}')
print(f'Years in Splits:   {sorted(ss["YEAR"].unique())}')

In [ ]:
kp.head(3)

In [ ]:
ss.head(3)

## 2. Select Relevant Columns

From **KenPom Barttorvik**: identity + efficiency + 3PT stats + outcomes

From **Shooting Splits**: detailed shot-type shares and FG% (offense & defense)

In [ ]:
# ---- KenPom columns ----
kp_cols = [
    # Identity
    'YEAR', 'TEAM ID', 'TEAM', 'CONF',
    # Tournament outcome
    'SEED', 'ROUND',
    # Overall performance
    'W', 'L', 'WIN%',
    'KADJ EM',   # KenPom adjusted efficiency margin
    'KADJ O',    # adjusted offensive efficiency
    'KADJ D',    # adjusted defensive efficiency
    'BARTHAG',   # win probability vs average D1 team
    'BADJ EM',   # Barttorvik adjusted efficiency margin
    # 3-point offense
    '3PT%',      # offensive 3PT field goal %
    '3PTR',      # offensive 3PT attempt rate (share of shots that are 3s)
    # 3-point defense
    '3PT%D',     # defensive 3PT FG% allowed
    '3PTRD',     # opponent 3PT attempt rate (share of opponent shots that are 3s)
    # Other shooting context
    'EFG%',      # effective FG% offense
    'EFG%D',     # effective FG% defense
    '2PT%',
    '2PT%D',
    '2PTR',
    '2PTRD',
    # Pace
    'KADJ T',    # adjusted tempo
    # Talent / experience
    'TALENT',
    'EXP',
    'WAB',       # wins above bubble
    'ELITE SOS', # strength of schedule
]

# ---- Shooting Splits columns ----
ss_cols = [
    'YEAR', 'TEAM ID',
    # 3s offense
    'THREES FG%',
    'THREES SHARE',
    # 3s defense
    'THREES FG%D',
    'THREES D SHARE',
    # 2s offense
    'CLOSE TWOS FG%',
    'CLOSE TWOS SHARE',
    'FARTHER TWOS FG%',
    'FARTHER TWOS SHARE',
    # Dunks
    'DUNKS FG%',
    'DUNKS SHARE',
]

kp_sel = kp[kp_cols].copy()
ss_sel = ss[ss_cols].copy()

print('KenPom selected:', kp_sel.shape)
print('Splits selected:', ss_sel.shape)

## 3. Merge Datasets

Join on `YEAR` + `TEAM ID`. Use a left join on KenPom (base) since Shooting Splits starts from 2010.

In [ ]:
df = kp_sel.merge(ss_sel, on=['YEAR', 'TEAM ID'], how='left')

print(f'Combined shape: {df.shape}')
print(f'Years covered: {sorted(df["YEAR"].unique())}')
print(f'\nMissing values (Shooting Splits cols only):')
ss_only_cols = [c for c in ss_cols if c not in ['YEAR', 'TEAM ID']]
print(df[ss_only_cols].isna().sum())

## 4. Clean & Validate

In [ ]:
# ROUND encoding in the source data:
#   0  = did not make the tournament
#   68 = First Four (play-in game)
#   64 = Round of 64 (lost in first round)
#   32 = Round of 32
#   16 = Sweet 16
#   8  = Elite 8
#   4  = Final Four
#   2  = Championship game
#   1  = Champion
print('ROUND distribution:')
print(df['ROUND'].value_counts().sort_index())

In [ ]:
# Map ROUND to an ordinal tournament_round (higher = further)
round_map = {0: 0, 68: 1, 64: 2, 32: 3, 16: 4, 8: 5, 4: 6, 2: 7, 1: 8}
df['tournament_round'] = df['ROUND'].map(round_map)

# Binary: made the main tournament (64-team bracket or better)
df['made_tournament'] = (df['ROUND'] >= 64).astype(int)

# Binary: reached at least Sweet 16
df['reached_s16'] = (df['tournament_round'] >= 4).astype(int)

# Rename WIN% for convenience
df.rename(columns={'WIN%': 'win_pct'}, inplace=True)

print(f'Tournament teams: {df["made_tournament"].sum()} / {len(df)}')
print(f'Sweet 16+ teams:  {df["reached_s16"].sum()} / {len(df)}')

In [ ]:
# Check for duplicate team-year rows
dupes = df.duplicated(subset=['YEAR', 'TEAM ID'])
print(f'Duplicate YEAR+TEAM ID rows: {dupes.sum()}')

# Check missing values across all columns
missing = df.isna().sum()
print('\nColumns with missing values:')
print(missing[missing > 0])

In [ ]:
# Drop rows where core 3PT stats are missing (2008-2009 have no Shooting Splits)
# Keep all rows — the KenPom 3PT% / 3PTR cols cover the full range
# Flag rows missing the Splits-sourced columns
df['has_shot_splits'] = df['THREES FG%'].notna().astype(int)

print('Rows with shot splits data:', df['has_shot_splits'].sum())
print('Rows without shot splits:  ', (df['has_shot_splits'] == 0).sum())

## 5. Feature Engineering

Create derived features useful for analysis.

In [ ]:
# 3PT volume category (how three-point-heavy is the offense?)
df['three_rate_category'] = pd.qcut(
    df['3PTR'],
    q=4,
    labels=['Low (Q1)', 'Mid-Low (Q2)', 'Mid-High (Q3)', 'High (Q4)']
)

# 3PT efficiency category
df['three_pct_category'] = pd.qcut(
    df['3PT%'],
    q=4,
    labels=['Low (Q1)', 'Mid-Low (Q2)', 'Mid-High (Q3)', 'High (Q4)']
)

# Net 3PT differential: how much better is your 3PT% than your opponents'?
df['three_pct_net'] = df['3PT%'] - df['3PT%D']

# Net 3PT rate differential: do you take more 3s than your opponents do against you?
df['three_rate_net'] = df['3PTR'] - df['3PTRD']

# 3PT value score: combines volume and efficiency (approximation)
# Points per 3PT attempt relative to league-average 2PT value (1.0 pts/attempt)
df['three_value'] = df['3PT%'] * 3 * (df['3PTR'] / 100)

print(df[['3PT%', '3PTR', '3PT%D', '3PTRD',
           'three_pct_net', 'three_rate_net', 'three_value']].describe().round(2))

### Derive 3PA and 3PM (season totals)

Since raw counts aren't in the source data, we back-calculate from four-factor rates:

```
Points = FGA × (2×EFG% + FTR×FT%)   →   FGA = K_TEMPO × PPPO / (2×EFG% + FTR×FT%)
3PA = 3PTR × FGA_season
3PM = 3PT% × 3PA
```

In [ ]:
# Pull extra columns needed for derivation that weren't in kp_cols
kp_extra = kp[['YEAR', 'TEAM ID', 'K TEMPO', 'FT%', 'FTR', 'PPPO', 'GAMES']].copy()
df = df.merge(kp_extra, on=['YEAR', 'TEAM ID'], how='left')

# All rate/pct columns stored as percentages → convert to decimals
efg     = df['EFG%']  / 100
ftr     = df['FTR']   / 100
ft_pct  = df['FT%']   / 100
thr_r   = df['3PTR']  / 100
thr_pct = df['3PT%']  / 100
two_r   = df['2PTR']  / 100
two_pct = df['2PT%']  / 100

# Back-calculate FGA from scoring identity:
#   Points = FGA × (2×EFG% + FTR×FT%)
#   PPPO   = Points / possession  →  FGA/poss = PPPO / (2×EFG% + FTR×FT%)
fga_per_game = df['K TEMPO'] * df['PPPO'] / (2 * efg + ftr * ft_pct)
fga_season   = fga_per_game * df['GAMES']

# 3-point attempts and makes (season totals)
df['3PA'] = (thr_r   * fga_season).round().astype(int)
df['3PM'] = (thr_pct * df['3PA']).round().astype(int)

# 2-point attempts and makes (season totals)
df['2PA'] = (fga_season - df['3PA']).round().astype(int)
df['2PM'] = (two_pct * df['2PA']).round().astype(int)

print(df[['TEAM', 'YEAR', 'GAMES', '3PTR', '3PT%', '3PA', '3PM', '2PA', '2PM']].head(10).to_string(index=False))
print(f"\n3PA per game (avg): {(df['3PA'] / df['GAMES']).mean():.1f}")
print(f"3PM per game (avg): {(df['3PM'] / df['GAMES']).mean():.1f}")
print(f"2PA per game (avg): {(df['2PA'] / df['GAMES']).mean():.1f}")
print(f"2PM per game (avg): {(df['2PM'] / df['GAMES']).mean():.1f}")

## 6. Final Dataset Overview

In [ ]:
print('=== Combined NCAA Dataset ===')
print(f'Rows: {df.shape[0]}  |  Columns: {df.shape[1]}')
print(f'Years: {df["YEAR"].min()} – {df["YEAR"].max()}')
print(f'Teams per year (avg): {df.groupby("YEAR").size().mean():.1f}')
print()
print('Key 3PT columns:')
three_cols = ['3PT%', '3PTR', '3PT%D', '3PTRD',
              'three_pct_net', 'three_rate_net', 'three_value',
              'THREES FG%', 'THREES SHARE', 'THREES FG%D', 'THREES D SHARE']
print(df[three_cols].describe().round(3))
print()
print('Key success columns:')
success_cols = ['win_pct', 'KADJ EM', 'BARTHAG', 'SEED',
                'tournament_round', 'made_tournament', 'reached_s16']
print(df[success_cols].describe().round(3))

In [ ]:
df.head(5)

## 7. Save Combined Dataset

In [ ]:
out_path = os.path.join(DATA_DIR, 'combined_ncaa.csv')
df.to_csv(out_path, index=False)
print(f'Saved to: {out_path}')
print(f'Shape: {df.shape}')
print(f'\nFinal columns:')
for c in df.columns:
    print(f'  {c}')